# Giving LLMs Access to Tools: AI Agent Foundations

**What are we building?** An AI assistant that can look up real data from Salesforce — not by hallucinating, but by actually calling an API. This is the foundational mechanic behind AI agents: **tool calling**.

**What is tool calling?** When an LLM has access to tools, it can decide *on its own* that it needs external data to answer a question. Instead of responding with text, it responds with a structured request: "call this function with these arguments." Your code executes the function, sends the result back, and the LLM gives an informed answer.

```
User: "Tell me about Edge Communications"
  → LLM decides: I need to call get_account("Edge Communications")
    → Your code runs the function, queries Salesforce
      → LLM receives the data, responds with a real answer
```

## Part 1: Connect to Our Data Sources

First we connect to Salesforce using `simple_salesforce` — a lightweight Python library that talks to the Salesforce REST API using standard SOQL queries. No custom Apex required.

In [40]:
import os
from dotenv import load_dotenv
import json
import requests
from gradio.components.markdown import Markdown
from simple_salesforce import Salesforce
import _sqlite3
load_dotenv(override=True)
sf_client = Salesforce(password=os.getenv('SF_PASSWORD'),
                       username=os.getenv('SF_USERNAME'),
                       security_token=os.getenv('SF_SECURITY_TOKEN'),
                       domain=os.getenv('DOMAIN'))


/Users/brooksjohnson/bbc-ai-engineering/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [41]:
# verify connection
result = sf_client.query("SELECT Name, Id, Industry, Phone, BillingCity, BillingState FROM Account LIMIT 10")

accounts = []
for account in result['records']:
    accounts.append({
        "name": account["Name"],
        "id": account["Id"],
        "industry": account.get("Industry"),
        "phone": account.get("Phone"),
        "billing_city": account.get("BillingCity"),
        "billing_state": account.get("BillingState"),
    })
accounts[0]

{'name': 'Edge Communications',
 'id': '001gL000000XCDhQAO',
 'industry': 'Electronics',
 'phone': '(512) 757-6000',
 'billing_city': 'Austin',
 'billing_state': 'TX'}

In [42]:
def get_account(account_name):
    result = sf_client.query(
        f"SELECT Name, Id, Industry, Phone, BillingCity, BillingState "
        f"FROM Account "
        f"WHERE Name LIKE '%{account_name}%' "
        f"LIMIT 5"
    )
    accounts = []
    for account in result['records']:
        accounts.append({
            "name": account["Name"],
            "id": account["Id"],
            "industry": account.get("Industry"),
            "phone": account.get("Phone"),
            "billing_city": account.get("BillingCity"),
            "billing_state": account.get("BillingState"),
        })
    return {"accounts": accounts, "count": len(accounts)}

get_account("Edge")

{'accounts': [{'name': 'Edge Communications',
   'id': '001gL000000XCDhQAO',
   'industry': 'Electronics',
   'phone': '(512) 757-6000',
   'billing_city': 'Austin',
   'billing_state': 'TX'}],
 'count': 1}

## Part 2: Define the Tool Schema

To give an LLM a tool, you have to **describe it in JSON**. This schema tells the model: what the function is called, what it does, what arguments it takes, and which are required.

This is the OpenAI function-calling format — and it's become the de facto standard. Ollama, Azure OpenAI, Anthropic, Google, and others all support this same structure.

In [17]:
# Define the tool schema — this is the JSON structure that tells the LLM
# what tools it can use, what arguments they take, and when to use them.
# This is the OpenAI function-calling format, which Ollama also supports.

tools = [
    {
        "type": "function",  # Always "function" — the only type currently supported
        "function": {
            "name": "get_account",  # Must match your actual Python function name
            "description": "Look up a Salesforce Account by name. Use this when the user asks about a company, customer, or account.",
            "parameters": {
                "type": "object",  # Parameters are always described as a JSON object
                "properties": {
                    "account_name": {
                        "type": "string",
                        "description": "The full or partial name of the account to search for"
                    }
                },
                "required": ["account_name"]  # Which parameters the LLM must provide
            }
        }
    }
]

# That's a LOT of boilerplate for one function with one parameter.
# Imagine defining 5-10 tools this way. Now you see why SDKs exist.
print(json.dumps(tools, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "get_account",
      "description": "Look up a Salesforce Account by name. Use this when the user asks about a company, customer, or account.",
      "parameters": {
        "type": "object",
        "properties": {
          "account_name": {
            "type": "string",
            "description": "The full or partial name of the account to search for"
          }
        },
        "required": [
          "account_name"
        ]
      }
    }
  }
]


In [27]:
prompt = [{"role": "system", "content": "You are a helpful business assistant designed to help users manage accounts.Always resond in english If you not know the answer say you"},
    {"role": "user", "content": "Tell me about the account named Edge Communications"},
]

## Part 3: Make the Tool Call

Here's where it gets interesting. We're going to use the **OpenAI Python SDK** to talk to Ollama.

**Why the OpenAI SDK for Ollama?** OpenAI's API format has become the industry standard — like how USB-C became the standard charging port. Most LLM providers (including Ollama) now expose an OpenAI-compatible endpoint. This means **one SDK works with many models**:

- **Ollama** (local) → `base_url="http://localhost:11434/v1"`
- **Azure OpenAI** (cloud) → `base_url="https://your-resource.openai.azure.com/"`  
- **OpenAI** (cloud) → default, no base_url needed

Same code, different endpoint. That's the key insight: **tool calling is a standard, not a vendor feature.**

In [36]:
ollama_url = "http://localhost:11434/v1"
from openai import OpenAI
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
response = ollama.chat.completions.create(model='qwen2.5:7b', messages=prompt, tools=tools)

# When the LLM wants to use a tool, .content is None — the response is in .tool_calls instead
message = response.choices[0].message
print("Content:", message.content)
print("Tool calls:", message.tool_calls)

if message.tool_calls:
    tool_call = message.tool_calls[0]  # Get the first tool call from the list
    function_name = tool_call.function.name  # e.g. "get_account"
    arguments = json.loads(tool_call.function.arguments)  # JSON string -> dict
    print(f"\nLLM wants to call: {function_name}")
    print(f"With arguments: {arguments}")

    # Map the string name to our actual Python function
    available_tools = {"get_account": get_account}
    tool_result = available_tools[function_name](**arguments)
    print(f"\nTool returned: {json.dumps(tool_result, indent=2)}")

prompt.append(message)
prompt.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(tool_result)
})

Content: 
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_5w08pqfu', function=Function(arguments='{"account_name":"Edge Communications"}', name='get_account'), type='function', index=0)]

LLM wants to call: get_account
With arguments: {'account_name': 'Edge Communications'}

Tool returned: {
  "accounts": [
    {
      "name": "Edge Communications",
      "id": "001gL000000XCDhQAO",
      "industry": "Electronics",
      "phone": "(512) 757-6000",
      "billing_city": "Austin",
      "billing_state": "TX"
    }
  ],
  "count": 1
}


### The Tool Call Loop

When we send a message with `tools=tools`, the LLM has a choice: respond with text, or request a tool call. If it requests a tool call, `message.content` will be `None` and `message.tool_calls` will contain the function name and arguments.

Our job is to:
1. Detect the tool call
2. Execute the actual Python function
3. Send the result back to the LLM as a `role: "tool"` message
4. Call the LLM again — now it has real data and responds with a natural language answer

In [39]:
from IPython.display import Markdown, display

response = ollama.chat.completions.create(model='qwen2.5:7b', messages=prompt, tools=tools)
display(Markdown(response.choices[0].message.content))

The account named Edge Communications has the following details:

- Industry: Electronics
- Phone: (512) 757-6000
- Billing City: Austin
- Billing State: TX

The ID of this account is 001gL000000XCDhQAO.